# Bài 2: cài ptLasso, glmtrans trên Kaggle (PD-04)

Notebook chạy bằng kernel **Python**. Nó tự cài một bản R riêng bằng micromamba (conda-forge), clone đúng commit đã push rồi gọi `scripts/r/check_r_baselines.R`: cài hai gói, chạy ví dụ binomial trong manual CRAN và ghi `sessionInfo()`.
Đây là kiểm tra môi trường trên dữ liệu mô phỏng của hai gói, chưa chạy baseline trên dữ liệu đề tài và chưa ép đúng k (M6).

Mỗi phiên:
1. Language: **Python** (mặc định). Accelerator: None. Bật **Internet** (cần để clone, tải R và tải gói từ CRAN).
2. Không cần gắn Input.
3. Kiểm `REPO_COMMIT` ở ô cấu hình là commit đã push.
4. Save Version → Save & Run All. Kết quả: `/kaggle/working/r_baselines_results.zip`.

Kết quả lỗi vẫn có giá trị: zip ghi lỗi cụ thể của từng bước để giảng viên quyết phương án thay thế.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, tarfile, time, urllib.request

REPO_URL = 'https://github.com/datachain-uit/CN-23520899-LuanNT.git'
REPO_BRANCH = 'rq2-mvp'
REPO_COMMIT = '67bbbe1'              # commit đã push chứa scripts/r/check_r_baselines.R

REPO_DIR = Path('/tmp/CN-23520899-LuanNT')
OUT_DIR = Path('/kaggle/working/r_baselines')
LIB_DIR = Path('/tmp/r_baselines_lib')    # ngoài /kaggle/working để R và gói đã cài không lọt vào output
MAMBA_DIR = Path('/tmp/micromamba')
R_ENV = Path('/tmp/r_env')
MICROMAMBA_URL = 'https://micro.mamba.pm/api/micromamba/linux-64/latest'
# R mới và các phụ thuộc nặng lấy bản biên dịch sẵn; ptLasso, glmtrans cài từ CRAN trong script R.
CONDA_PACKAGES = ['r-base>=4.4', 'r-glmnet', 'r-ggplot2', 'r-gridextra', 'r-matrix', 'r-caret', 'r-foreach', 'r-doparallel', 'r-assertthat', 'r-formatr']
print(sys.version)

In [ ]:
# Gate: clone đúng commit đã push (lỗi ở đây dừng ngay: chưa có gì để lưu).
assert REPO_COMMIT, 'Điền REPO_COMMIT ở ô cấu hình'
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--quiet', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--quiet', REPO_COMMIT], cwd=REPO_DIR, check=True)
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
pushed = subprocess.check_output(['git', 'branch', '-r', '--contains', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert pushed, f'{commit_hash} không có trên remote: kết quả sẽ không truy vết được'
SCRIPT = REPO_DIR / 'scripts/r/check_r_baselines.R'
assert SCRIPT.is_file(), f'Commit {commit_hash} không có {SCRIPT.name}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('COMMIT', commit_hash)

In [ ]:
# Cài R riêng bằng micromamba; lỗi được ghi lại, không dừng, để ô cuối vẫn lưu zip. Có thể mất vài phút.
setup_error = None
micromamba = MAMBA_DIR / 'bin' / 'micromamba'
t0 = time.time()
try:
    MAMBA_DIR.mkdir(parents=True, exist_ok=True)
    archive_mm = MAMBA_DIR / 'micromamba.tar.bz2'
    urllib.request.urlretrieve(MICROMAMBA_URL, archive_mm)
    with tarfile.open(archive_mm) as tar:
        tar.extract('bin/micromamba', MAMBA_DIR)
    os.chmod(micromamba, 0o755)
    with open(OUT_DIR / 'setup_r.log', 'w', encoding='utf-8') as log:
        subprocess.run([str(micromamba), 'create', '--yes', '--quiet', '--prefix', str(R_ENV), '--override-channels', '--channel', 'conda-forge', *CONDA_PACKAGES],
                       stdout=log, stderr=subprocess.STDOUT, check=True, env={**os.environ, 'MAMBA_ROOT_PREFIX': str(MAMBA_DIR)})
    print(subprocess.check_output([str(R_ENV / 'bin' / 'Rscript'), '--version'], text=True, stderr=subprocess.STDOUT).strip())
except Exception as exc:
    setup_error = repr(exc)
    (OUT_DIR / 'setup_error.txt').write_text(setup_error, encoding='utf-8')
    print('LỖI CÀI R:', setup_error)
print(f'Cài R: {(time.time() - t0) / 60:.1f} phút')

In [ ]:
# Chạy script R; không raise ở đây để kết quả luôn được lưu ở ô sau.
script_status = None
if setup_error is None:
    env = {k: v for k, v in os.environ.items() if k not in ('LD_LIBRARY_PATH', 'R_HOME', 'PYTHONPATH')}
    env.update({'PATH': f"{R_ENV / 'bin'}:{env['PATH']}", 'R_BASELINE_OUT': str(OUT_DIR), 'R_BASELINE_LIB': str(LIB_DIR), 'REPO_COMMIT_HASH': commit_hash})
    t0 = time.time()
    with open(OUT_DIR / 'run_stdout.log', 'w', encoding='utf-8') as out, open(OUT_DIR / 'run_stderr.log', 'w', encoding='utf-8') as err:
        script_status = subprocess.run([str(R_ENV / 'bin' / 'Rscript'), str(SCRIPT)], stdout=out, stderr=err, env=env, cwd=REPO_DIR).returncode
    print('SCRIPT EXIT', script_status, f'sau {(time.time() - t0) / 60:.1f} phút')
    print('\n'.join((OUT_DIR / 'run_stdout.log').read_text(encoding='utf-8', errors='replace').splitlines()[-40:]))

In [ ]:
# Lưu kết quả rồi mới báo lỗi, để lỗi cài đặt vẫn tải về được.
archive = shutil.make_archive('/kaggle/working/r_baselines_results', 'zip', root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name)
print('ARCHIVE', archive, os.path.getsize(archive), 'byte')
session_info = OUT_DIR / 'sessionInfo.txt'
if session_info.exists():
    print(session_info.read_text(encoding='utf-8', errors='replace'))
if setup_error is not None:
    raise RuntimeError(f'Không cài được R: {setup_error}. Xem setup_r.log trong zip')
if script_status != 0:
    raise RuntimeError(f'Script R thoát mã {script_status}: xem r_baseline_check.csv và run_stderr.log trong zip')
print('R_BASELINES_OK')